In [41]:
import numpy as np
import librosa
import audioflux as af
import soundfile as sf
import matplotlib.pyplot as plt
import sklearn

Load the file and split into grains for analysis

In [93]:
sr=48000
y, sr = af.read("..\..\corpus\metro_sample_2\input.wav", samplate=sr) # sr is 48kHz

grain_duration = 0.1 # in s
grain_size = int(grain_duration * sr)
n_grains_in_source = int(len(y) // grain_size)
grains = [i*grain_size for i in range(n_grains_in_source)]

# since BFT is based on the FFT, the number of bins needs to be half that of a power of 2
bft_obj = af.BFT(num=2049, samplate=sr, radix2_exp=12, slide_length=grain_size,
               data_type=af.type.SpectralDataType.MAG,
               scale_type=af.type.SpectralFilterBankScaleType.LINEAR)
spec_arr = bft_obj.bft(y)
spec_arr = np.abs(spec_arr)
spectral_obj = af.Spectral(num=bft_obj.num,
                           fre_band_arr=bft_obj.get_fre_band_arr())
n_time = spec_arr.shape[-1]  
spectral_obj.set_time_length(n_time)
rms_arr = spectral_obj.rms(spec_arr)
centroid_arr = spectral_obj.centroid(spec_arr)

n_clusters=4
x = np.array([[i, j] for i, j in zip(rms_arr, centroid_arr)])
kmeans = sklearn.cluster.KMeans(n_clusters=n_clusters, n_init='auto', random_state=0).fit(x)

# plt.scatter(centroid_arr, rms_arr, edgecolors='k', alpha=0.7, c=kmeans.labels_)
# plt.figure(figsize=(10, 7))
# plt.ylabel("Spectral Root Mean Square")
# plt.xlabel("Spectral Centroid")
# plt.grid(True, linestyle='--', alpha=0.6)
# plt.show()

dict_clusters = {}
for idx, lab in enumerate(kmeans.labels_):
    dict_clusters[lab] = dict_clusters.get(lab, [])
    dict_clusters[lab].append(idx)


In [94]:
transition_matrix = np.zeros((n_clusters, n_clusters))
labels = kmeans.labels_
prev_label = labels[0]
for label_idx in range(1, len(labels)):
    transition_matrix[prev_label][labels[label_idx]] += 1
    prev_label = labels[label_idx]
total_freq = np.sum(transition_matrix)

transition_matrix_fin = []#np.zeros((n_clusters, n_clusters))
for i in transition_matrix:
    row = i / np.sum(i)
    transition_matrix_fin.append(row)
transition_matrix_fin = np.array(transition_matrix_fin)
transition_matrix_fin

array([[0.61029412, 0.        , 0.34558824, 0.04411765],
       [0.00403226, 0.78225806, 0.00806452, 0.20564516],
       [0.19918699, 0.01626016, 0.60162602, 0.18292683],
       [0.00980392, 0.16339869, 0.16013072, 0.66666667]])

In [95]:
output_buffer = np.array([])
curr_label = 0
n_iterations = 100
seed = 100
s = np.random.seed(seed)

for i in range(n_iterations):

    next_label = np.random.choice(range(n_clusters), p=transition_matrix_fin[curr_label])
    print(next_label)
    grain_idx = np.random.choice(dict_clusters[next_label]) # uniform sampling from a cluster
    grain_y_idx = grains[grain_idx]
    grain = y[grain_y_idx:grain_y_idx+grain_size]
    grain = grain * np.hanning(grain_size)
    output_buffer = np.concatenate([output_buffer, grain])
    curr_label = next_label
# output_buffer / np.max(np.abs(output_buffer))
sf.write(f"..\..\corpus\metro_sample_2\output\mm_grain_dur{grain_duration}_startcluster_{curr_label}_randseed_{seed}.wav", output_buffer, samplerate=sr)

0
0
2
0
2
2
2
2
0
0
2
0
2
3
3
3
0
2
0
0
0
0
0
0
2
0
0
2
2
0
2
0
2
2
2
2
2
3
3
3
3
3
3
3
2
2
2
2
2
3
3
1
1
1
1
1
1
3
3
3
2
0
2
3
1
3
3
1
1
3
3
3
3
3
3
3
3
3
1
2
3
3
3
3
3
2
2
0
0
0
2
2
2
1
1
3
3
3
3
3
